### Transformar Datos de "Attendances"
1. Extraer la "Fecha" del campo "check_in" y crear una nueva columna llamada "date"
2. Extrae la "hora" de los campos "check_in" y "check_out" y crear unas nuevas columnas llamadas "check_in_time" y "check_out_time"
3. Asignar al campo "status" los siguientes valores descriptivos:<br>
(1: Present, 2: Absent, 3: Remote, 4: Late, 5: Vacation, 6: On Leave)
4. Escribir los datos "transformados" en el schema "silver"

In [0]:
df = spark.table("zenviro.bronze.attendances_py")
display(df)

#### 1. Extraer la "Fecha" del campo "check_in" y crear una nueva columna llamada "date"

In [0]:
from pyspark.sql.functions import date_format
df_extracted = (
    df
    .select(
        "attendance_id",
        "employee_id",
        date_format("check_in", "yyyy-MM-dd").cast("date").alias("date"),
        "check_in",
        "check_out",
        "status"
    )
)
display(df_extracted)

#### 2. Extrae la "hora" de los campos "check_in" y "check_out" y crear unas nuevas columnas llamadas "check_in_time" y "check_out_time"

In [0]:
from pyspark.sql.functions import date_format
df_extracted_time = (
    df_extracted
    .select(
        "attendance_id",
        "employee_id",
        "date",
        date_format("check_in", "hh:mm:ss").alias("check_in_time"),
        date_format("check_out", "hh:mm:ss").alias("check_out_time"),
        "status"
    )
)
display(df_extracted_time)

#### 3. Asignar al campo "status" los siguientes valores descriptivos:<br>
(1: Present, 2: Absent, 3: Remote, 4: Late, 5: Vacation, 6: On Leave)

In [0]:
from pyspark.sql.functions import when, col
df_mapped = (
    df_extracted_time
    .select(
        col("attendance_id").cast("int").alias("attendance_id"),
        col("employee_id").cast("int").alias("employee_id"),
        "date",
        "check_in_time",
        "check_out_time",
        when(df_extracted_time.status == 1, "Present")
        .when(df_extracted_time.status == 2, "Absent")
        .when(df_extracted_time.status == 3, "Remote")
        .when(df_extracted_time.status == 4, "Late")
        .when(df_extracted_time.status == 5, "Vacation")
        .when(df_extracted_time.status == 6, "On Leave")
        .alias("status")
    )
)
display(df_mapped)

#### 4. Escribir los datos "transformados" en el schema "silver"

In [0]:
df_mapped.writeTo("zenviro.silver.attendances_py").createOrReplace()

In [0]:
display(spark.table("zenviro.silver.attendances_py"))